# Renderizar Motion Graphics (HyperFrames) en Colab (sin apt, vía pip)

Esta versión usa **pip** para FFmpeg (imageio-ffmpeg) y Chromium (Playwright), y **nvm** para Node.js.
Ideal para subir el notebook a GitHub y ejecutarlo en Colab sin instalar nada en tu PC.

Pasos:
1. Ejecuta las celdas en orden.
2. Sube un `.zip` de tu proyecto (con `index.html`).
3. Descarga el `output.mp4`.

In [ ]:
# 1) Node.js 22 via nvm (sin apt)
!curl -o- https://raw.githubusercontent.com/nvm-sh/nvm/v0.39.7/install.sh | bash
!export NVM_DIR="$HOME/.nvm" && . "$NVM_DIR/nvm.sh" && nvm install 22 && nvm use 22 && node --version && npm --version

In [ ]:
# 2) FFmpeg y Chromium via pip
!pip install -q imageio-ffmpeg playwright
!playwright install chromium
import imageio_ffmpeg, playwright
print('ffmpeg bin:', imageio_ffmpeg.get_ffmpeg_exe())

In [ ]:
# 3) Configurar variables de entorno para HyperFrames
import os, subprocess, glob
ffmpeg_bin = imageio_ffmpeg.get_ffmpeg_exe()
os.environ['HYPERFRAMES_FFMPEG_PATH'] = ffmpeg_bin
# Chromium de Playwright
chromium_candidates = glob.glob('/root/.cache/ms-playwright/chromium-*/chrome-linux/chrome') + glob.glob('/home/*/.cache/ms-playwright/chromium-*/chrome-linux/chrome')
chromium_bin = chromium_candidates[0] if chromium_candidates else None
if chromium_bin:
    os.environ['CHROME_PATH'] = chromium_bin
print('FFMPEG:', os.environ.get('HYPERFRAMES_FFMPEG_PATH'))
print('CHROME:', os.environ.get('CHROME_PATH'))

In [ ]:
# 4) Subir el proyecto (zip)
from google.colab import files
uploaded = files.upload()
print('Subido:', list(uploaded.keys()))

In [ ]:
# 5) Descomprimir
import zipfile, os
zip_name = list(uploaded.keys())[0]
os.makedirs('/content/project', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/project')
entries = os.listdir('/content/project')
if len(entries) == 1 and os.path.isdir(os.path.join('/content/project', entries[0])):
    inner = os.path.join('/content/project', entries[0])
    for f in os.listdir(inner):
        os.rename(os.path.join(inner, f), os.path.join('/content/project', f))
    os.rmdir(inner)
print('Contenido:', os.listdir('/content/project'))

In [ ]:
# 6) Renderizar usando nvm + node
import os
nvm_dir = os.path.expanduser('~/.nvm')
cmd = f'''export NVM_DIR="{nvm_dir}" && . "$NVM_DIR/nvm.sh" && nvm use 22 && cd /content/project && npx hyperframes render --output /content/output.mp4'''
!{cmd}

In [ ]:
# 7) Descargar el video
from google.colab import files
files.download('/content/output.mp4')